<h1 style="color:DodgerBlue">Индивидуальный проект: учёт заказов</h1>


<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Учёт заказов товаров и услуг

**Вариант задания № 9** — базовый класс Order и его наследники OnlineOrder, PhysicalOrder, SpecializedOrder.


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Проект моделирует учёт заказов товаров и услуг на C# согласно варианту 9 из файла «Варианты заданий индивидуальных проектов (Трек 1).pdf».

Базовый абстрактный класс **Order** хранит номер заказа (OrderId), дату создания (CreationDate), итоговую сумму (TotalAmount) и закрытый список позиций. Методы AddItem, RemoveItem и CalculateTotal добавляют позиции, удаляют их и рассчитывают итоговую стоимость. После изменения списка сумма пересчитывается автоматически.

**Item** представляет одну позицию: название, цену за единицу и количество. Стоимость позиции равна цене, умноженной на количество.

| Вид заказа | Дополнительные данные | Переопределение | Собственный метод |
|---|---|---|---|
| OnlineOrder | CustomerEmail, DeliveryMethod | AddItem добавляет позицию и сообщает способ доставки | ShowDeliveryInfo |
| PhysicalOrder | DeliveryAddress | RemoveItem сообщает о возврате при успешном удалении | ShowDeliveryAddress |
| SpecializedOrder | SpecialConditions, DiscountPercent | CalculateTotal учитывает процентную скидку | ShowSpecialConditions |

В качестве конкретного специального условия выбрана скидка: итог равен исходной сумме × (1 − процент скидки / 100). Скидка допускается от 0 до 100 процентов. Информация о доставке выводится в консоль; отдельный тариф доставки в этой версии не рассчитывается.

Для каждого наследника создаются **два объекта**. Массив Order[] позволяет выполнять операции через общий тип и демонстрирует полиморфизм. В реализации также показаны инкапсуляция, наследование и абстракция.

Данные демонстрации заданы заранее. Добавление, удаление, повторное удаление отсутствующей позиции и расчёт скидки показаны в выводе программы. Отрицательная цена, неположительное количество, пустые обязательные строки и null вместо позиции отклоняются проверками.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;

// Позиция заказа: наименование, цена за единицу и количество.
class Item
{
    public string Name { get; }
    public decimal Price { get; }
    public int Quantity { get; }

    public Item(string name, decimal price, int quantity = 1)
    {
        if (string.IsNullOrWhiteSpace(name))
            throw new ArgumentException("Название позиции не должно быть пустым.", nameof(name));
        if (price < 0)
            throw new ArgumentOutOfRangeException(nameof(price), "Цена не может быть отрицательной.");
        if (quantity <= 0)
            throw new ArgumentOutOfRangeException(nameof(quantity), "Количество должно быть положительным.");

        Name = name;
        Price = price;
        Quantity = quantity;
    }

    public decimal GetCost()
    {
        return Price * Quantity;
    }
}

// Общее описание заказа; создаём объекты конкретных наследников.
abstract class Order
{
    // Список меняется только через методы заказа.
    private readonly List<Item> items = new List<Item>();

    public int OrderId { get; }
    public DateTime CreationDate { get; }
    public decimal TotalAmount { get; protected set; }
    public int ItemCount { get { return items.Count; } }

    protected Order(int orderId, DateTime creationDate)
    {
        if (orderId <= 0)
            throw new ArgumentOutOfRangeException(nameof(orderId), "Номер заказа должен быть положительным.");

        OrderId = orderId;
        CreationDate = creationDate;
    }

    public virtual decimal CalculateTotal()
    {
        decimal total = 0;
        foreach (Item item in items)
            total += item.GetCost();

        TotalAmount = total;
        return TotalAmount;
    }

    public virtual void AddItem(Item item)
    {
        if (item == null)
            throw new ArgumentNullException(nameof(item));

        items.Add(item);
        // Вызов виртуального метода учитывает вид заказа.
        CalculateTotal();
    }

    public virtual bool RemoveItem(Item item)
    {
        if (item == null)
            throw new ArgumentNullException(nameof(item));

        bool removed = items.Remove(item);
        if (removed)
            CalculateTotal();

        return removed;
    }
}

class OnlineOrder : Order
{
    public string CustomerEmail { get; }
    public string DeliveryMethod { get; }

    public OnlineOrder(int orderId, DateTime creationDate,
                       string customerEmail, string deliveryMethod)
        : base(orderId, creationDate)
    {
        if (string.IsNullOrWhiteSpace(customerEmail))
            throw new ArgumentException("Email не должен быть пустым.", nameof(customerEmail));
        if (string.IsNullOrWhiteSpace(deliveryMethod))
            throw new ArgumentException("Укажите способ доставки.", nameof(deliveryMethod));

        CustomerEmail = customerEmail;
        DeliveryMethod = deliveryMethod;
    }

    public override void AddItem(Item item)
    {
        base.AddItem(item);
        Console.WriteLine($"Онлайн-заказ #{OrderId}: добавлено «{item.Name}»; доставка — {DeliveryMethod}.");
    }

    public void ShowDeliveryInfo()
    {
        Console.WriteLine($"Заказ #{OrderId}: {DeliveryMethod}; email клиента: {CustomerEmail}.");
    }
}

class PhysicalOrder : Order
{
    public string DeliveryAddress { get; }

    public PhysicalOrder(int orderId, DateTime creationDate, string deliveryAddress)
        : base(orderId, creationDate)
    {
        if (string.IsNullOrWhiteSpace(deliveryAddress))
            throw new ArgumentException("Адрес доставки не должен быть пустым.", nameof(deliveryAddress));

        DeliveryAddress = deliveryAddress;
    }

    public override bool RemoveItem(Item item)
    {
        bool removed = base.RemoveItem(item);
        if (removed)
            Console.WriteLine($"Физический заказ #{OrderId}: возврат «{item.Name}» зарегистрирован; позиция удалена.");
        else
            Console.WriteLine($"Физический заказ #{OrderId}: позиции «{item.Name}» нет; возврат не оформлен.");

        return removed;
    }

    public void ShowDeliveryAddress()
    {
        Console.WriteLine($"Адрес доставки заказа #{OrderId}: {DeliveryAddress}.");
    }
}

class SpecializedOrder : Order
{
    public string SpecialConditions { get; }
    public decimal DiscountPercent { get; }

    public SpecializedOrder(int orderId, DateTime creationDate,
                            string specialConditions, decimal discountPercent)
        : base(orderId, creationDate)
    {
        if (string.IsNullOrWhiteSpace(specialConditions))
            throw new ArgumentException("Укажите специальные условия.", nameof(specialConditions));
        if (discountPercent < 0 || discountPercent > 100)
            throw new ArgumentOutOfRangeException(nameof(discountPercent), "Скидка должна быть от 0 до 100 процентов.");

        SpecialConditions = specialConditions;
        DiscountPercent = discountPercent;
    }

    public override decimal CalculateTotal()
    {
        // Каждый расчёт начинается с исходной стоимости позиций.
        // Поэтому повторный вызов не применяет скидку второй раз.
        decimal subtotal = base.CalculateTotal();
        TotalAmount = decimal.Round(subtotal * (1 - DiscountPercent / 100m),
                                    2, MidpointRounding.AwayFromZero);
        return TotalAmount;
    }

    public void ShowSpecialConditions()
    {
        string discount = DiscountPercent.ToString("0.##", CultureInfo.InvariantCulture);
        Console.WriteLine($"Условия заказа #{OrderId}: {SpecialConditions}; скидка {discount}%.");
    }
}

// Демонстрационные значения заданы заранее, без Console.ReadLine().
DateTime creationDate = new DateTime(2026, 10, 8);

OnlineOrder onlineOrder1 = new OnlineOrder(101, creationDate, "buyer1@example.com", "курьер");
OnlineOrder onlineOrder2 = new OnlineOrder(102, creationDate, "buyer2@example.com", "пункт выдачи");
PhysicalOrder physicalOrder1 = new PhysicalOrder(201, creationDate, "Учебная улица, дом 1");
PhysicalOrder physicalOrder2 = new PhysicalOrder(202, creationDate, "Учебная улица, дом 2");
SpecializedOrder specializedOrder1 = new SpecializedOrder(301, creationDate, "Скидка постоянному клиенту", 10m);
SpecializedOrder specializedOrder2 = new SpecializedOrder(302, creationDate, "Скидка на услуги", 20m);

// Через ссылки типа Order вызываются реализации реальных объектов.
Order[] orders = {
    onlineOrder1, onlineOrder2, physicalOrder1, physicalOrder2,
    specializedOrder1, specializedOrder2
};

Console.WriteLine("Добавление позиций:");
orders[0].AddItem(new Item("Мышь", 750m, 2));
orders[0].AddItem(new Item("Клавиатура", 2000m));
orders[1].AddItem(new Item("Флешка", 600m, 3));

Item monitor = new Item("Монитор", 12000m);
orders[2].AddItem(monitor);
orders[2].AddItem(new Item("Кабель", 500m, 2));
orders[3].AddItem(new Item("Наушники", 2500m));
orders[4].AddItem(new Item("Комплект мебели", 20000m));
orders[5].AddItem(new Item("Установка оборудования", 5000m, 2));

Console.WriteLine();
Console.WriteLine("Удаление позиции и информация о возврате:");
orders[2].RemoveItem(monitor);
// Второе удаление показывает, что отсутствующая позиция не меняет сумму.
orders[2].RemoveItem(monitor);

Console.WriteLine();
Console.WriteLine("Дополнительная информация:");
onlineOrder1.ShowDeliveryInfo();
onlineOrder2.ShowDeliveryInfo();
physicalOrder1.ShowDeliveryAddress();
physicalOrder2.ShowDeliveryAddress();
specializedOrder1.ShowSpecialConditions();
specializedOrder2.ShowSpecialConditions();

Console.WriteLine();
Console.WriteLine("Суммы заказов — полиморфный вызов CalculateTotal():");
foreach (Order order in orders)
{
    decimal total = order.CalculateTotal();
    string amount = total.ToString("F2", CultureInfo.InvariantCulture);
    Console.WriteLine($"#{order.OrderId} | {order.GetType().Name} | {order.CreationDate:dd.MM.yyyy} | позиций: {order.ItemCount} | сумма: {amount} руб.");
}



Добавление позиций:
Онлайн-заказ #101: добавлено «Мышь»; доставка — курьер.
Онлайн-заказ #101: добавлено «Клавиатура»; доставка — курьер.
Онлайн-заказ #102: добавлено «Флешка»; доставка — пункт выдачи.

Удаление позиции и информация о возврате:
Физический заказ #201: возврат «Монитор» зарегистрирован; позиция удалена.
Физический заказ #201: позиции «Монитор» нет; возврат не оформлен.

Дополнительная информация:
Заказ #101: курьер; email клиента: buyer1@example.com.
Заказ #102: пункт выдачи; email клиента: buyer2@example.com.
Адрес доставки заказа #201: Учебная улица, дом 1.
Адрес доставки заказа #202: Учебная улица, дом 2.
Условия заказа #301: Скидка постоянному клиенту; скидка 10%.
Условия заказа #302: Скидка на услуги; скидка 20%.

Суммы заказов — полиморфный вызов CalculateTotal():
#101 | OnlineOrder | 08.10.2026 | позиций: 2 | сумма: 3500.00 руб.
#102 | OnlineOrder | 08.10.2026 | позиций: 1 | сумма: 1800.00 руб.
#201 | PhysicalOrder | 08.10.2026 | позиций: 1 | сумма: 1000.00 руб.
#